# Learn Transformers by Teaching a Model to Add Numbers

This notebook walks through a **complete, minimal Transformer** built from scratch in PyTorch.

We train it on strings like `<bos>23+45=68<eos>` and watch it learn to **generate the correct sum** at inference time.

> **Goal:** By the end, you should be able to trace data from raw text → token IDs → embeddings → attention → logits → generated digits.

---
## 0. The big picture

A **Transformer** is a stack of blocks that repeatedly:
1. Lets every token **look at previous tokens** (self-attention)
2. Applies a small **feed-forward network** per token
3. Uses **residual connections** and **layer normalization** for stable training

GPT-style models are **decoder-only**: they predict the **next token** given all previous tokens. Training is self-supervised — we don't need labels beyond the text itself.

```
Training example:  <bos> 2 7 + 1 5 = 6 8 <eos>
Input  (x):         <bos> 2 7 + 1 5 = 6 8
Target (y):          2   7 + 1 5 = 6 8 <eos>
                     ↑ predict this at each step
```

At inference we feed `<bos>12+34=` and generate `46<eos>` one character at a time.

---
## 1. Environment setup

Run from the project folder:

```bash
pip install -r requirements.txt
```

In [ ]:
import math
import random
from pathlib import Path

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

from data import (
    BOS_TOKEN, EOS_TOKEN, PAD_ID, VOCAB, STOI, ITOS,
    DataConfig, encode, decode, format_example, build_datasets,
)
from model import AdditionTransformer, ModelConfig, count_parameters

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
print("Device:", device)
print("Vocabulary:", VOCAB)

---
## 2. Tokenization — turning text into numbers

Neural networks consume numbers, not characters. We build a tiny vocabulary:

| Token | Meaning |
|-------|--------|
| `<pad>` | Padding (batch alignment) |
| `<bos>` | Beginning of sequence |
| `<eos>` | End of sequence |
| `0`…`9` | Digit characters |
| `+`, `=` | Operators |

Each token maps to a unique integer ID (index into an embedding table later).

In [ ]:
a, b = 23, 45
text = format_example(a, b)
ids = encode(text)

print("String:", text)
print("Token IDs:", ids)
print("Decoded:", decode(ids))
print("Ground truth sum:", a + b)

### Causal language modeling targets

For each position `t`, the model sees tokens `[0..t]` and must predict token `t+1`.

This is why Transformers for text generation use a **causal mask** — position `i` cannot attend to future position `j > i`.

In [ ]:
tokens = ids
x = tokens[:-1]
y = tokens[1:]

print(f"{'Position':<10} {'Input (x)':<12} {'Target (y)':<12}")
print("-" * 36)
for i, (xi, yi) in enumerate(zip(x, y)):
    print(f"{i:<10} {ITOS[xi]:<12} {ITOS[yi]:<12}")

---
## 3. Embeddings + positional encoding

### Token embeddings
Each token ID is looked up in an `nn.Embedding(vocab_size, d_model)` matrix → a vector of size `d_model`.

### Positional encoding
Attention is **permutation-equivariant** without position info — `2+3` and `3+2` would look identical.

We add sinusoidal vectors (from the original Transformer paper) so each position has a unique signature:

$$PE_{(pos, 2i)} = \sin\left(\frac{pos}{10000^{2i/d}}\right), \quad PE_{(pos, 2i+1)} = \cos\left(\frac{pos}{10000^{2i/d}}\right)$$

In [ ]:
d_model = 64
max_seq_len = 24

position = torch.arange(max_seq_len).unsqueeze(1)
div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
pe = torch.zeros(max_seq_len, d_model)
pe[:, 0::2] = torch.sin(position * div_term)
pe[:, 1::2] = torch.cos(position * div_term)

plt.figure(figsize=(10, 4))
plt.imshow(pe[:20].T, aspect="auto", cmap="RdBu")
plt.xlabel("Position in sequence")
plt.ylabel("Embedding dimension")
plt.title("Sinusoidal positional encoding (first 20 positions)")
plt.colorbar()
plt.show()

---
## 4. Scaled dot-product attention (the core idea)

For each token we compute three vectors from its hidden state:
- **Query (Q):** "What am I looking for?"
- **Key (K):** "What do I contain?"
- **Value (V):** "What information do I pass if selected?"

Attention weights for token `i` over keys `j`:

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}} + \text{mask}\right) V$$

**Multi-head** means we run several attention operations in parallel on smaller subspaces, then concatenate.

In [ ]:
seq_len = 8
d_k = 16

Q = torch.randn(seq_len, d_k)
K = torch.randn(seq_len, d_k)
V = torch.randn(seq_len, d_k)

scores = Q @ K.T / math.sqrt(d_k)
causal = torch.tril(torch.ones(seq_len, seq_len))
scores = scores.masked_fill(causal == 0, float("-inf"))
weights = F.softmax(scores, dim=-1)
output = weights @ V

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(scores.detach(), cmap="viridis")
axes[0].set_title("Raw scores (masked upper triangle)")
axes[1].imshow(weights.detach(), cmap="Blues")
axes[1].set_title("Attention weights (causal)")
plt.show()

print("Output shape:", output.shape, "= (seq_len, d_k)")

---
## 5. One Transformer block

```
x ──► LayerNorm ──► Multi-Head Attention ──► (+) ──► LayerNorm ──► FFN ──► (+) ──► out
│                                              ▲                           ▲
└──────────────────────────────────────────────┘                           │
└──────────────────────────────────────────────────────────────────────────┘
                         (residual connections)
```

Our model stacks **2 blocks** by default, then a final LayerNorm and linear **LM head** to predict the next token.

In [ ]:
model_cfg = ModelConfig(d_model=96, n_heads=4, n_layers=3, d_ff=192, max_seq_len=16)
model = AdditionTransformer(model_cfg).to(device)
print(f"Parameters: {count_parameters(model):,}")
print(model)

### Forward pass on a single example

Output logits have shape `(batch, sequence, vocab_size)` — at each position, a score for every possible next token.

In [ ]:
sample_text = format_example(12, 34)
sample_ids = torch.tensor([encode(sample_text)[:-1]], device=device)

with torch.no_grad():
    logits = model(sample_ids)

print("Input string (without last token):", decode(sample_ids[0].tolist()))
print("Logits shape:", tuple(logits.shape))

last_pos_logits = logits[0, -1]
top5 = torch.topk(last_pos_logits, 5)
print("\nTop-5 predictions after '=' (last position shown):")
for score, idx in zip(top5.values, top5.indices):
    print(f"  {ITOS[idx.item()]:>5}  score={score.item():.2f}")

---
## 6. Training loop

We minimize **cross-entropy** between predicted logits and the true next token, ignoring padded positions.

Data is synthetic: random `(a, b)` pairs in `[0, 99]`.

In [ ]:
def masked_cross_entropy(logits, labels, loss_mask):
    loss = F.cross_entropy(
        logits.reshape(-1, logits.size(-1)),
        labels.reshape(-1),
        reduction="none",
    )
    loss = loss.view(labels.shape)
    return (loss * loss_mask).sum() / loss_mask.sum().clamp(min=1.0)


data_cfg = DataConfig(max_number=9, train_size=10_000, val_size=500)
train_ds, val_ds = build_datasets(data_cfg)
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=64)

model = AdditionTransformer(model_cfg).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)

epochs = 20
history = {"train_loss": [], "val_loss": []}

for epoch in range(1, epochs + 1):
    model.train()
    total, tokens = 0.0, 0.0
    for batch in tqdm(train_loader, desc=f"Epoch {epoch}"):
        input_ids = batch["input_ids"].to(device)
        labels = batch["labels"].to(device)
        loss_mask = batch["loss_mask"].to(device)
        key_padding_mask = input_ids == PAD_ID

        logits = model(input_ids, key_padding_mask=key_padding_mask)
        loss = masked_cross_entropy(logits, labels, loss_mask)

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total += loss.item() * loss_mask.sum().item()
        tokens += loss_mask.sum().item()

    model.eval()
    val_total, val_tokens = 0.0, 0.0
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch["input_ids"].to(device)
            labels = batch["labels"].to(device)
            loss_mask = batch["loss_mask"].to(device)
            logits = model(input_ids, key_padding_mask=input_ids == PAD_ID)
            loss = masked_cross_entropy(logits, labels, loss_mask)
            val_total += loss.item() * loss_mask.sum().item()
            val_tokens += loss_mask.sum().item()

    history["train_loss"].append(total / tokens)
    history["val_loss"].append(val_total / val_tokens)
    print(f"Epoch {epoch}: train={history['train_loss'][-1]:.4f} val={history['val_loss'][-1]:.4f}")

In [ ]:
plt.plot(history["train_loss"], label="train")
plt.plot(history["val_loss"], label="val")
plt.xlabel("Epoch")
plt.ylabel("Cross-entropy loss")
plt.title("Training curve")
plt.legend()
plt.show()

---
## 7. Inference — autoregressive generation

Generation loop (greedy decoding):
1. Feed prompt tokens
2. Take argmax of logits at last position → next token
3. Append token, repeat until `<eos>` or max length

This is exactly how GPT produces text, just on a tiny vocabulary.

In [ ]:
from data import EOS_ID

def predict(a, b):
    prompt = f"{BOS_TOKEN}{a}+{b}="
    prompt_ids = torch.tensor([encode(prompt)], device=device)
    out = model.generate(prompt_ids, max_new_tokens=8, eos_id=EOS_ID)
    text = decode(out[0].tolist())
    predicted = text.split("=")[-1] if "=" in text else "?"
    return text, predicted, str(a + b)


test_cases = [(3, 5), (7, 2), (9, 9), (4, 6), (0, 8), (1, 1)]
print(f"{'Prompt':<12} {'Model':<8} {'Expected':<8} OK")
print("-" * 40)
for a, b in test_cases:
    full, pred, exp = predict(a, b)
    ok = "✓" if pred == exp else "✗"
    print(f"{a}+{b}=       {pred:<8} {exp:<8} {ok}")

---
## 8. Step-by-step generation trace

Watch the model append one digit at a time.

In [ ]:
a, b = 27, 15
prompt = f"{BOS_TOKEN}{a}+{b}="
generated = torch.tensor([encode(prompt)], device=device)

print("Step-by-step greedy decoding:")
for step in range(6):
    with torch.no_grad():
        logits = model(generated)
    next_id = logits[0, -1].argmax().item()
    token = ITOS[next_id]
    print(f"  step {step}: current='{decode(generated[0].tolist())}' → next='{token}'")
    generated = torch.cat([generated, torch.tensor([[next_id]], device=device)], dim=1)
    if token == EOS_TOKEN:
        break

---
## 9. Mental map: this demo vs. GPT / LLaMA

| Component | This demo | Large LLM |
|-----------|-----------|----------|
| Tokenizer | Character-level | BPE / SentencePiece (subword) |
| Vocab | ~15 tokens | 32k–128k tokens |
| `d_model` | 64 | 4,096+ |
| Layers | 2 | 32–80+ |
| Task | Addition strings | General text |
| Training | Random sums | Trillions of tokens |
| Objective | Next-token CE | Same |

**The algorithm is the same.** Scale, data, and tokenizer complexity differ.

---

## 10. Exercises

1. Increase `--max-number` to 999 and add a third Transformer layer. How many epochs are needed?
2. Replace greedy decoding with **top-k sampling** (`torch.topk` + random choice).
3. Modify `MultiHeadSelfAttention` to return attention weights and plot them for `45+27=`.
4. Remove positional encoding — does the model still learn addition? Why or why not?
5. Read `model.py` side-by-side with [The Illustrated Transformer](https://jalammar.github.io/illustrated-transformer/).

---

## 11. Save checkpoint (optional)

The notebook model can be saved and loaded by `inference.py`.

In [ ]:
save_path = Path("checkpoints/addition_transformer.pt")
save_path.parent.mkdir(parents=True, exist_ok=True)
torch.save(
    {
        "model_state": model.state_dict(),
        "model_config": model_cfg.__dict__,
        "data_config": data_cfg.__dict__,
    },
    save_path,
)
print(f"Saved to {save_path}")
print("Run: python inference.py --a 27 --b 15")